# Large Language Models (BERT / GPT)

_Modern Deep Learning & AI_

**Predict the next word over and over. Do it on the whole internet and language understanding falls out.**

A large language model (LLM) is a giant Transformer trained on a simple game: guess the next word.

     Two famous styles. GPT reads left to right and predicts the next token. BERT sees the whole sentence and predicts masked (hidden) tokens.

---

This notebook is a practice scaffold for the **Large Language Models (BERT / GPT)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DecoderBlock(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
    def forward(self, x):
        T = x.size(1)
        mask = torch.triu(torch.ones(T, T), diagonal=1).bool()  # block future tokens
        a, _ = self.attn(self.n1(x), self.n1(x), self.n1(x), attn_mask=mask)
        x = x + a                                  # residual
        return x + self.ff(self.n2(x))             # residual

torch.manual_seed(0)
vocab, d, T = 50, 32, 6
emb = nn.Embedding(vocab, d)
block, head = DecoderBlock(d, 4), nn.Linear(d, vocab)
tokens = torch.randint(0, vocab, (1, T))           # tiny synthetic sequence
logits = head(block(emb(tokens)))                  # (1, T, vocab)
# next-token loss: predict token t+1 from positions up to t
loss = F.cross_entropy(logits[:, :-1].reshape(-1, vocab), tokens[:, 1:].reshape(-1))
print("logits shape:", logits.shape, "loss:", round(loss.item(), 3))

## Visualize the data & results

_After the real prompt The capital of France is, what is the next token at low vs high temperature?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# next-token distribution for the prompt "The capital of France is"
labels = ["Paris", "a", "located", "the", "home", "now"]
logits = np.array([9.1, 4.2, 3.8, 3.1, 2.4, 1.7])

def softmax_T(z, T):
    s = z / T; s -= s.max(); e = np.exp(s); return e / e.sum()

p_lo = softmax_T(logits, 0.7) * 100     # sharp
p_hi = softmax_T(logits, 1.5) * 100     # flatter

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
c = ["#7ee787"] + ["#4ea1ff"] * 5       # highlight the top token
axes[0].bar(labels, p_lo, color=c)
axes[0].set_title("next token at T = 0.7"); axes[0].set_ylabel("percent")
axes[1].bar(labels, p_hi, color="#ffb454")
axes[1].set_title("same logits at T = 1.5")
plt.show()